# 11.05 Anthropic File Search Middleware

`StateFileSearchMiddleware` 는 그래프 상태 안에 들어 있는 가상 파일들(예: `text_editor_files`, `memory_files`)을
**glob + grep** 으로 검색할 수 있게 해주는 Claude 네이티브 도구를 제공한다.

텍스트 에디터/메모리 미들웨어가 **파일을 만들고 편집** 한다면, 이 미들웨어는 그 위에서 **파일을 찾고 읽는다**.

## 학습 목표

- `StateFileSearchMiddleware(state_key=...)` 로 검색 대상 파일 저장소를 선택한다
- `StateClaudeTextEditorMiddleware` 와 함께 써서 "쓰고 → 찾고 → 읽는" 루프를 완성한다
- `state_key="memory_files"` 로 메모리 쪽 파일도 검색 대상으로 돌린다

## 언제 쓰나

- 에이전트가 **수십~수백 개의 가상 파일**을 상태에 쌓아놓고 그 안을 탐색해야 할 때
- 이름만 기억나는 파일을 패턴(`**/*.md`)으로 찾거나, 특정 키워드를 포함한 파일만 골라야 할 때
- 메모리에 쌓인 과거 노트를 모델이 스스로 grep 해서 참조하게 하고 싶을 때

## 11.05.1 환경 설정

필요 패키지: `langchain`, `langchain-anthropic`. `.env`에 `ANTHROPIC_API_KEY`.

In [ ]:
# !pip install -U langchain langchain-anthropic

from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024)
model.invoke("ping").content[:50]

## 11.05.2 텍스트 에디터 + 파일 검색 조합

검색은 단독으로는 의미가 없다 — **파일이 먼저 상태에 있어야** 한다.
가장 흔한 조합은 `StateClaudeTextEditorMiddleware` 로 파일을 만들고 `StateFileSearchMiddleware` 로 찾는 패턴이다.

| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `state_key` | `"text_editor_files"` | 검색할 파일 dict 가 들어 있는 상태 키. `"memory_files"` 로 바꾸면 메모리 쪽을 검색 |

In [ ]:
from langchain.agents import create_agent
from langchain_anthropic.middleware import (
    StateClaudeTextEditorMiddleware,
    StateFileSearchMiddleware,
)

agent = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6", max_tokens=2048),
    tools=[],
    middleware=[
        StateClaudeTextEditorMiddleware(allowed_path_prefixes=["/docs"]),
        StateFileSearchMiddleware(state_key="text_editor_files"),
    ],
)

## 11.05.3 1단계 — 여러 파일을 상태에 심는다

모델에게 몇 개의 노트 파일을 `/docs/` 아래에 만들어 달라고 요청한다.
이후 검색 쿼리에서 이 파일들이 타깃이 된다.

In [ ]:
seed = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "아래 3개 파일을 /docs 아래에 만들어줘.\n"
            "- /docs/backend/python.md : 'FastAPI 팁: async def 라우트는 blocking IO 를 피할 것.'\n"
            "- /docs/backend/go.md : 'Go 팁: context.Context 로 취소를 전파하라.'\n"
            "- /docs/frontend/react.md : 'React 팁: useMemo 는 측정 후 적용.'\n"
        ),
    }]
})
print("생성된 파일:", list(seed.get("text_editor_files", {}).keys()))

## 11.05.4 2단계 — 같은 에이전트가 glob/grep 으로 검색

동일 스레드에서 이어 호출하면 이전 파일들이 상태에 남아 있다.
모델은 `StateFileSearchMiddleware` 가 제공하는 `glob` 과 `grep` 도구로 파일을 찾고 내용을 읽는다.

In [ ]:
# glob 패턴으로 backend 문서만 찾아 읽기
r1 = agent.invoke({
    "messages": seed["messages"] + [{
        "role": "user",
        "content": "/docs/backend/**/*.md 에 해당하는 파일만 골라 각 파일의 핵심 1줄 요약을 묶어 보여줘.",
    }],
    "text_editor_files": seed.get("text_editor_files", {}),
})
print("--- glob 결과 ---")
print(r1["messages"][-1].content[:500])

In [ ]:
# grep 패턴으로 키워드가 들어간 파일 찾기
r2 = agent.invoke({
    "messages": seed["messages"] + [{
        "role": "user",
        "content": "'취소' 또는 'cancel' 이 언급된 파일을 찾아 파일 경로와 문장을 인용해줘.",
    }],
    "text_editor_files": seed.get("text_editor_files", {}),
})
print("--- grep 결과 ---")
print(r2["messages"][-1].content[:500])

## 11.05.5 메모리 파일도 검색 대상으로

`state_key="memory_files"` 로 바꾸면 `StateClaudeMemoryMiddleware` 가 쌓아둔 메모를 검색할 수 있다.

> ⚠️ **중복 미들웨어 제약**: LangChain 1.2 의 `create_agent` 는 **같은 미들웨어 클래스의 중복 인스턴스를 거부**한다 (`AssertionError: Please remove duplicate middleware instances.`). 텍스트 에디터 파일과 메모리 파일 두 저장소를 동시에 검색하려면 한 번에 하나만 등록하거나, `StateFileSearchMiddleware` 를 서브클래싱해 별도 클래스로 분리해야 한다.

In [ ]:
from langchain_anthropic.middleware import StateClaudeMemoryMiddleware

# LangChain 1.2 의 create_agent 는 같은 미들웨어 클래스의 중복 인스턴스를 거부한다
# (`AssertionError: Please remove duplicate middleware instances.`).
# state_key 마다 별도 인스턴스를 등록하는 대신 하나만 사용하거나 서브클래싱하라.
agent_dual = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6", max_tokens=2048),
    tools=[],
    middleware=[
        StateClaudeTextEditorMiddleware(allowed_path_prefixes=["/docs"]),
        StateClaudeMemoryMiddleware(),
        StateFileSearchMiddleware(state_key="text_editor_files"),
        # StateFileSearchMiddleware(state_key="memory_files"),  # ← 같은 클래스 추가 등록 불가
    ],
)
print("agent_dual 준비 완료 — text_editor_files 검색 가능 (memory_files 는 별도 서브클래스 필요)")

## 11.05.6 언제 일반 vectorstore 대신 이 미들웨어를 쓰나

- **정확한 파일명/패턴** 이 중요하고, 의미 검색이 아닌 **리터럴 grep** 이면 충분할 때
- 문서 수가 **수십~수백 개** 수준이고 매 턴 임베딩을 다시 할 비용이 아까울 때
- 파일이 **현재 세션에서 방금 만들어진** 것이라 벡터 인덱스가 없을 때

대규모 코퍼스나 의미 검색이 필요하면 `07_integration` 의 RAG 계열 노트북으로 넘어간다.

## 체크리스트

- [ ] `state_key` 가 실제로 채워지는 파일 저장소와 일치하는가 (`text_editor_files` / `memory_files`)
- [ ] 검색 전에 파일을 **동일 스레드**에 쌓아뒀는가 (스레드가 다르면 비어 있다)
- [ ] glob/grep 으로 충분한지, 의미 검색이 필요한지 먼저 점검했는가

## 다음 노트북

- `06_bedrock_prompt_caching.ipynb` — AWS Bedrock 경유 프롬프트 캐시

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/middleware/anthropic